In [7]:
# =================================================================================================
# 1. IMPORT LIBRARIES & PERSIAPAN DATASET UTUH
# =================================================================================================
import os
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight

print("\n[INFO] Libraries berhasil dimuat!")


[INFO] Libraries berhasil dimuat!


In [8]:
# --- KONFIGURASI GLOBAL ---
DATASET_DIR = pathlib.Path("data_cropped") # Pastikan foldernya benar
IMG_SIZE = 224 
BATCH_SIZE = 16 
LEARNING_RATE = 0.0005
EPOCHS = 75
K_FOLDS = 5

# --- LOAD PATH DATASET ---
all_image_paths = list(DATASET_DIR.glob('*/*.png'))
all_image_paths = [str(path) for path in all_image_paths]

class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
all_image_labels = [label_to_index[pathlib.Path(path).parent.name] for path in all_image_paths]

X = np.array(all_image_paths)
y = np.array(all_image_labels)

print(f"[INFO] Total dataset siap K-Fold: {len(X)} gambar")

# --- FUNGSI PIPELINE DATA (MASKING & AUGMENTASI) ---
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=4, expand_animations=False)
    img = tf.cast(img, tf.float32)
    # Masking
    rgb = img[..., :3]
    alpha = img[..., 3:4] / 255.0
    img_fixed = rgb * alpha
    # Resize & Normalisasi
    img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
    img_final = img_resized / 255.0
    img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
    
    label_one_hot = tf.one_hot(label, depth=len(class_names_list))
    return img_final, label_one_hot

# Augmentasi Spasial yang Aman (Rotasi, Flip, Geser)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1) 
])

def augment(image, label):
    return data_augmentation(image, training=True), label

[INFO] Total dataset siap K-Fold: 904 gambar


In [9]:
# =================================================================================================
# 2. DEFINISI FOCAL LOSS KUSTOM (ANTI-ERROR)
# =================================================================================================
def categorical_focal_loss(alpha, gamma=1.0):
    """
    Fungsi Kustom Focal Loss
    alpha: list dari bobot kelas (untuk mengatasi imbalance)
    gamma: parameter fokus (2.0 adalah standar industri)
    """
    alpha_tensor = tf.constant(alpha, dtype=tf.float32)
    
    def focal_loss(y_true, y_pred):
        # Mencegah nilai log(0)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        # Cross Entropy dasar
        cross_entropy = -y_true * tf.math.log(y_pred)
        # Menghitung seberapa sulit tebakannya -> (1 - p)^gamma
        weight = alpha_tensor * tf.math.pow(1.0 - y_pred, gamma)
        # Terapkan hukuman
        loss = weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
        
    return focal_loss

In [10]:
# =================================================================================================
# 3. ARSITEKTUR CUSTOM CNN (RESIDUAL + SE-BLOCK ATTENTION)
# =================================================================================================

def se_block(x, filters, ratio=8):
    """Squeeze-and-Excitation Block (Attention Mechanism)"""
    # Squeeze: Meringkas info spasial
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, filters))(se)
    # Excitation: Menentukan filter mana yang penting
    se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Terapkan penguatan/pelemahan sinyal
    return layers.Multiply()([x, se])

def residual_block(x, filters, downsample=False):
    """Residual Block dengan Skip Connection dan Attention"""
    shortcut = x
    strides = (2, 2) if downsample else (1, 1)

    # Jalur Utama
    x = layers.Conv2D(filters, (3, 3), strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, (3, 3), strides=(1, 1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    
    # Masukkan Mata Batin (Attention)
    x = se_block(x, filters)

    # Penyesuaian Jalur Tol jika ukuran berubah
    if downsample or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Gabungkan Jalur Utama dengan Jalur Tol
    x = layers.Add()([shortcut, x])
    x = layers.Activation('relu')(x)
    return x

def build_advanced_uv_cnn():
    """Merakit Keseluruhan Otak Model"""
    inputs = tf.keras.Input(shape=(224, 224, 3), name="Input_Layer")
    
    # Layer Pembuka
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Tumpukan Blok Residual (ResNet Style)
    x = residual_block(x, 64, downsample=True) 
    x = residual_block(x, 128, downsample=True) 
    x = residual_block(x, 256, downsample=True) 
    
    # Inovasi Dual-Pooling Anda
    max_pool = layers.GlobalMaxPooling2D(name="Peak_Brightness_Detector")(x)
    avg_pool = layers.GlobalAveragePooling2D(name="Spatial_Extent_Detector")(x)
    merged = layers.Concatenate(name="Feature_Fusion")([max_pool, avg_pool])
    
    # Kepala Klasifikasi Diperbesar
    x = layers.Dense(256, activation='relu')(merged)
    x = layers.Dropout(0.3)(x) # Beban latihan 30%
    outputs = layers.Dense(4, activation='softmax', name="Output_Classification")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Advanced_ResNet_SE_CNN")
    return model

# Print summary sekali saja di awal
dummy_model = build_advanced_uv_cnn()
print("="*60)
print("ARSITEKTUR ADVANCED MODEL BERHASIL DIBANGUN")
print("="*60)
dummy_model.summary()

ARSITEKTUR ADVANCED MODEL BERHASIL DIBANGUN


Model: "Advanced_ResNet_SE_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 224, 224,  │        864 │ Input_Layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        128 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 112, 112,  │          0 │ activation_7[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 56, 56,    │     18,432 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_11[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 56, 56,    │     36,864 │ activation_8[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 1, 1, 64)  │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1, 1, 8)   │        512 │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 56, 56,    │      2,048 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1, 1, 64)  │        512 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_13[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Multiply)          │ 64)               │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 56, 56,    │          0 │ batch_normalizat

 Total params: 1,364,452 (5.20 MB)

 Trainable params: 1,361,700 (5.19 MB)

 Non-trainable params: 2,752 (10.75 KB)

In [12]:
# =================================================================================================
# 4. MULAI PERULANGAN K-FOLD CROSS VALIDATION
# =================================================================================================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
global_accuracies = []

for fold_no, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    print("\n" + "★"*60)
    print(f"🚀 MEMULAI TRAINING FOLD KE-{fold_no}")
    print("★"*60)
    
    X_train_paths, X_val_paths = X[train_index], X[val_index]
    y_train_labels, y_val_labels = y[train_index], y[val_index]
    
    # --- A. HITUNG ALPHA (CLASS WEIGHTS) OTOMATIS ---
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)
    # Ubah format menjadi array float agar bisa masuk ke rumus Focal Loss
    alpha_list = [float(w) for w in weights] 
    
    # --- B. BUAT DATASET ---
    train_ds = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train_labels))
    train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(len(X_train_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_labels))
    val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    test_ds = val_ds # Aliasing untuk evaluasi
    
    # --- C. COMPILE MODEL DENGAN FOCAL LOSS ---
    tf.keras.backend.clear_session()
    model = build_advanced_uv_cnn()
    
    # Merakit Focal Loss dengan nilai Alpha dinamis lipatan ini, dan Gamma 2.0
    loss_function = categorical_focal_loss(alpha=alpha_list, gamma=1.0)
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), 
                  loss=loss_function, 
                  metrics=['accuracy'])

    current_model_path = f'best_advanced_cnn_fold{fold_no}.keras'
    callbacks_list = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
        ModelCheckpoint(current_model_path, monitor='val_accuracy', save_best_only=True, verbose=0)
    ]

    # --- D. TRAINING (TIDAK MENGGUNAKAN class_weight LAGI) ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_list,
        verbose=1
    )

    # --- E. EVALUASI AKHIR ---
    print(f"\n[INFO] Melakukan Evaluasi Fold {fold_no}...")

    try:
        model.load_weights(current_model_path)
    except:
        pass # Gunakan bobot terakhir jika file gagal dimuat

    Y_pred_probs = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(Y_pred_probs, axis=1) 
    y_true_onehot = np.concatenate([y for x, y in test_ds], axis=0)
    y_true = np.argmax(y_true_onehot, axis=1)

    class_names = ['Kelas 1 (1 PPB)', 'Kelas 2 (2 PPB)', 'Kelas 3 (3 PPB)', 'Kelas 4 (4 PPB)']

    test_acc = accuracy_score(y_true, y_pred)
    global_accuracies.append(test_acc * 100)
    
    confidence_scores = np.max(Y_pred_probs, axis=1)
    avg_conf = np.mean(confidence_scores)

    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
    print("\n" + "=" * 60)
    print(f"📋 CLASSIFICATION REPORT FOLD {fold_no}")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_str = str(cm.tolist())

    plot_folder = "history_plots_customCNN" 
    os.makedirs(plot_folder, exist_ok=True)
    current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    plot_filename = f"{current_time_str}_Fold{fold_no}_CM_Acc{test_acc*100:.2f}.png" 
    plot_filepath = os.path.join(plot_folder, plot_filename)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 14})
    plt.title(f'Confusion Matrix Fold {fold_no} (Acc: {test_acc*100:.2f}%)')
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(plot_filepath, dpi=300, bbox_inches='tight')
    plt.close() 

    # --- F. VISUALISASI KURVA LOSS & AKURASI ---
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    plt.title(f'Kurva Akurasi Fold {fold_no} (Acc: {test_acc*100:.2f}%)', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Akurasi')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title(f'Kurva Focal Loss Fold {fold_no}', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Alpha-Balanced Focal Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()

    folder_name = "visualizations_CUSTOM"
    os.makedirs(folder_name, exist_ok=True) 
    
    file_name = f"{current_time_str}_Fold{fold_no}_Acc{test_acc*100:.2f}.png"
    save_path = os.path.join(folder_name, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    # --- G. SIMPAN LOG KE CSV ---
    LOG_FILE_PATH = "experiment_log_custom_cnn.csv"

    log_data = {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Fold_No": fold_no,
        "Model": "Advanced CNN (ResNet + SE + FocalLoss)",
        "Total_Params": model.count_params(),
        "Learning_Rate": LEARNING_RATE,
        "Use_Focal_Loss": "Yes (Gamma=2.0)",
        "Accuracy": round(test_acc, 4),
        "Avg_Confidence": round(avg_conf, 4),
        "Confusion_Matrix": cm_str
    }

    for idx, label in enumerate(class_names):
        metrics = report[label] 
        log_data[f"C{idx+1}_Prec"] = round(metrics['precision'], 4)
        log_data[f"C{idx+1}_Rec"] = round(metrics['recall'], 4)
        log_data[f"C{idx+1}_F1"] = round(metrics['f1-score'], 4)

    df_new_log = pd.DataFrame([log_data])

    if not os.path.exists(LOG_FILE_PATH):
        df_new_log.to_csv(LOG_FILE_PATH, index=False)
    else:
        df_new_log.to_csv(LOG_FILE_PATH, mode='a', header=False, index=False)



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🚀 MEMULAI TRAINING FOLD KE-1
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Epoch 1/75
46/46 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.3237 - loss: 5.4130 - val_accuracy: 0.2155 - val_loss: 1.0382 - learning_rate: 5.0000e-04
Epoch 2/75
46/46 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.3084 - loss: 1.8316 - val_accuracy: 0.2155 - val_loss: 1.0433 - learning_rate: 5.0000e-04
Epoch 3/75
46/46 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.3444 - loss: 1.1844 - val_accuracy: 0.2155 - val_loss: 1.0503 - learning_rate: 5.0000e-04
Epoch 4/75
46/46 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.3790 - loss: 0.9288 - val_accuracy: 0.2155 - val_loss: 1.0475 - learning_rate: 5.0000e-04
Epoch 5/75
46/46 ━━━━━━━━━━━━━━━━━━━━ 50s 977ms/step - accuracy: 0.3859 - loss: 0.8978 - val_accuracy: 0.2155 - val_loss: 1.0223 - learning_rate: 5.0000e-04
Epoch 6/75
46/46 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.3679 - loss

In [13]:
# --- REKAPITULASI AKHIR ---
print("\n" + "="*60)
print("🏆 KESIMPULAN HASIL 5-FOLD (ADVANCED MODEL)")
print("="*60)
for i, acc in enumerate(global_accuracies):
    print(f"Fold {i+1}: {acc:.2f}%")
print("-" * 30)
print(f"RATA-RATA AKURASI KESELURUHAN: {np.mean(global_accuracies):.2f}% (+/- {np.std(global_accuracies):.2f}%)")
print("="*60)


🏆 KESIMPULAN HASIL 5-FOLD (ADVANCED MODEL)
Fold 1: 59.12%
Fold 2: 65.75%
Fold 3: 63.54%
Fold 4: 56.35%
Fold 5: 60.00%
------------------------------
RATA-RATA AKURASI KESELURUHAN: 60.95% (+/- 3.32%)


In [ ]:
# =================================================================================================
# 1. IMPORT LIBRARIES & PERSIAPAN DATASET
# =================================================================================================
import os
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight

print("\n[INFO] Libraries berhasil dimuat!")

# --- KONFIGURASI GLOBAL ---
DATASET_DIR = pathlib.Path("data_cropped") # Pastikan foldernya benar
IMG_SIZE = 224 
BATCH_SIZE = 32 
LEARNING_RATE = 0.0005 # Kembali ke titik manis (Sweet Spot) Anda
EPOCHS = 75
DROPOUT_RATE = 0.3
K_FOLDS = 5

# --- LOAD PATH DATASET ---
all_image_paths = list(DATASET_DIR.glob('*/*.png'))
all_image_paths = [str(path) for path in all_image_paths]

class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
all_image_labels = [label_to_index[pathlib.Path(path).parent.name] for path in all_image_paths]

X = np.array(all_image_paths)
y = np.array(all_image_labels)

print(f"[INFO] Total dataset siap K-Fold: {len(X)} gambar")

# --- FUNGSI PIPELINE DATA (MASKING & AUGMENTASI) ---
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=4, expand_animations=False)
    img = tf.cast(img, tf.float32)
    # Masking UV
    rgb = img[..., :3]
    alpha = img[..., 3:4] / 255.0
    img_fixed = rgb * alpha
    # Resize & Normalisasi
    img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
    img_final = img_resized / 255.0
    img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
    
    label_one_hot = tf.one_hot(label, depth=len(class_names_list))
    return img_final, label_one_hot

# Augmentasi Spasial yang Aman (Tanpa Zoom, Tanpa Brightness)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1) 
])

def augment(image, label):
    return data_augmentation(image, training=True), label

# =================================================================================================
# 2. DEFINISI FOCAL LOSS KUSTOM (Alpha & Gamma)
# =================================================================================================
def categorical_focal_loss(alpha, gamma=2.0):
    """
    Fungsi Kustom Focal Loss.
    Alpha akan diisi otomatis oleh Class Weights. Gamma disetel standar 2.0.
    """
    alpha_tensor = tf.constant(alpha, dtype=tf.float32)
    
    def focal_loss(y_true, y_pred):
        # Hindari log(0)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        # Cross Entropy dasar
        cross_entropy = -y_true * tf.math.log(y_pred)
        # Efek Gamma (Fokus pada soal sulit)
        weight = alpha_tensor * tf.math.pow(1.0 - y_pred, gamma)
        loss = weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
        
    return focal_loss

# =================================================================================================
# 3. ARSITEKTUR CUSTOM CNN DUAL-POOLING (VERSI RAMPING & STABIL)
# =================================================================================================
def build_custom_cnn_focal():
    inputs = tf.keras.Input(shape=(224, 224, 3), name="Input_Layer")
    
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Inovasi Dual-Pooling
    max_pool = layers.GlobalMaxPooling2D(name="Peak_Brightness")(x)
    avg_pool = layers.GlobalAveragePooling2D(name="Spatial_Extent")(x)
    merged = layers.Concatenate(name="Feature_Fusion")([max_pool, avg_pool])
    
    # Kepala Klasifikasi Sedang (256 Neuron, Dropout 0.3)
    x = layers.Dense(256, activation='relu')(merged)
    x = layers.Dropout(DROPOUT_RATE)(x)
    
    outputs = layers.Dense(4, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Custom_CNN_Dual_Focal")
    return model

# Tampilkan arsitektur
dummy_model = build_custom_cnn_focal()
print("="*60)
print("ARSITEKTUR CUSTOM CNN + FOCAL LOSS SIAP!")
print("="*60)
dummy_model.summary()

# =================================================================================================
# 4. MULAI PERULANGAN K-FOLD CROSS VALIDATION
# =================================================================================================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
global_accuracies = []

for fold_no, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    print("\n" + "★"*60)
    print(f"🚀 MEMULAI TRAINING FOLD KE-{fold_no}")
    print("★"*60)
    
    X_train_paths, X_val_paths = X[train_index], X[val_index]
    y_train_labels, y_val_labels = y[train_index], y[val_index]
    
    # --- A. HITUNG ALPHA (CLASS WEIGHTS) OTOMATIS ---
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)
    alpha_list = [float(w) for w in weights] # Ubah ke float untuk Focal Loss
    
    # --- B. BUAT DATASET ---
    train_ds = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train_labels))
    train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(len(X_train_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_labels))
    val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    test_ds = val_ds # Aliasing untuk evaluasi
    
    # --- C. COMPILE MODEL DENGAN FOCAL LOSS ---
    tf.keras.backend.clear_session()
    model = build_custom_cnn_focal()
    
    # Masukkan Alpha dinamis dan Gamma 2.0
    loss_function = categorical_focal_loss(alpha=alpha_list, gamma=1.0)
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), 
                  loss=loss_function, 
                  metrics=['accuracy'])

    current_model_path = f'best_custom_cnn_focal_fold{fold_no}.keras'
    callbacks_list = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-6, verbose=1),
        ModelCheckpoint(current_model_path, monitor='val_accuracy', save_best_only=True, verbose=0)
    ]

    # --- D. TRAINING (TIDAK MENGGUNAKAN class_weight LAGI) ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_list,
        verbose=1
    )

    # --- E. EVALUASI AKHIR FOLD ---
    print(f"\n[INFO] Melakukan Evaluasi Fold {fold_no}...")
    try:
        model.load_weights(current_model_path)
    except:
        pass

    Y_pred_probs = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(Y_pred_probs, axis=1) 
    y_true_onehot = np.concatenate([y for x, y in test_ds], axis=0)
    y_true = np.argmax(y_true_onehot, axis=1)

    class_names = ['Kelas 1 (1 PPB)', 'Kelas 2 (2 PPB)', 'Kelas 3 (3 PPB)', 'Kelas 4 (4 PPB)']

    test_acc = accuracy_score(y_true, y_pred)
    global_accuracies.append(test_acc * 100)
    
    confidence_scores = np.max(Y_pred_probs, axis=1)
    avg_conf = np.mean(confidence_scores)

    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
    print("\n" + "=" * 60)
    print(f"📋 CLASSIFICATION REPORT FOLD {fold_no}")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Confusion Matrix Plot
    cm = confusion_matrix(y_true, y_pred)
    cm_str = str(cm.tolist())

    plot_folder = "history_plots_customCNN" 
    os.makedirs(plot_folder, exist_ok=True)
    current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    plot_filepath = os.path.join(plot_folder, f"{current_time_str}_Fold{fold_no}_CM_Acc{test_acc*100:.2f}.png")

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 14})
    plt.title(f'Confusion Matrix Fold {fold_no} (Acc: {test_acc*100:.2f}%)')
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(plot_filepath, dpi=300, bbox_inches='tight')
    plt.close() 

    # Visulisasi Kurva (Training vs Validation)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    plt.title(f'Kurva Akurasi Fold {fold_no} (Acc: {test_acc*100:.2f}%)', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Akurasi')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title(f'Kurva Focal Loss Fold {fold_no}', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    folder_name = "visualizations_CUSTOM"
    os.makedirs(folder_name, exist_ok=True) 
    save_path = os.path.join(folder_name, f"{current_time_str}_Fold{fold_no}_Acc{test_acc*100:.2f}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    # Log ke CSV
    LOG_FILE_PATH = "experiment_log_custom_cnn.csv"
    log_data = {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Fold_No": fold_no, # Tambahan untuk tracking fold
        "Model": "Custom CNN (Dual-Pooling) + Focal Loss",
        "Total_Params": model.count_params(),
        "Learning_Rate": LEARNING_RATE,
        "Batch_Size": BATCH_SIZE, 
        "Dropout_Rate": DROPOUT_RATE,
        "Img_Size": IMG_SIZE,  
        "Use_Class_Weights": "Yes (Dynamic K-Fold)",
        "Accuracy": round(test_acc, 4),
        "Avg_Confidence": round(avg_conf, 4),
        "Confusion_Matrix": cm_str
    }
    for idx, label in enumerate(class_names):
        metrics = report[label] 
        log_data[f"C{idx+1}_Prec"] = round(metrics['precision'], 4)
        log_data[f"C{idx+1}_Rec"] = round(metrics['recall'], 4)
        log_data[f"C{idx+1}_F1"] = round(metrics['f1-score'], 4)

    df_new_log = pd.DataFrame([log_data])
    if not os.path.exists(LOG_FILE_PATH):
        df_new_log.to_csv(LOG_FILE_PATH, index=False)
    else:
        df_new_log.to_csv(LOG_FILE_PATH, mode='a', header=False, index=False)

# --- REKAPITULASI AKHIR ---
print("\n" + "="*60)
print("🏆 KESIMPULAN HASIL 5-FOLD (CUSTOM CNN + FOCAL LOSS)")
print("="*60)
for i, acc in enumerate(global_accuracies):
    print(f"Fold {i+1}: {acc:.2f}%")
print("-" * 30)
print(f"RATA-RATA AKURASI KESELURUHAN: {np.mean(global_accuracies):.2f}% (+/- {np.std(global_accuracies):.2f}%)")
print("="*60)


[INFO] Libraries berhasil dimuat!
[INFO] Total dataset siap K-Fold: 904 gambar
ARSITEKTUR CUSTOM CNN + FOCAL LOSS SIAP!


Model: "Custom_CNN_Dual_Focal"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 224, 224,  │        448 │ Input_Layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │         64 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 112, 112,  │      4,640 │ max_pooling2d_4[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 56, 56,    │     18,496 │ max_pooling2d_5[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 28, 28,    │     73,856 │ max_pooling2d_6[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        512 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 14, 14,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Peak_Brightness     │ (None, 128)       │          0 │ max_pooling2d_7[… │
│ (GlobalMaxPooling2… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Spatial_Extent      │ (None, 128)       │          0 │ max_pooling2d_7[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Feature_Fusion      │ (None, 256)       │          0 │ Peak_Brightness[… │
│ (Concatenate)       │                   │            │ Spatial_Extent[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │     65,792 │ Feature_Fusion[0

 Total params: 165,220 (645.39 KB)

 Trainable params: 164,740 (643.52 KB)

 Non-trainable params: 480 (1.88 KB)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🚀 MEMULAI TRAINING FOLD KE-1
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Epoch 1/75
23/23 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.2379 - loss: 6.9373 - val_accuracy: 0.0994 - val_loss: 1.0402 - learning_rate: 5.0000e-04
Epoch 2/75
23/23 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - accuracy: 0.3596 - loss: 3.8394 - val_accuracy: 0.0994 - val_loss: 1.0597 - learning_rate: 5.0000e-04
Epoch 3/75
23/23 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.3361 - loss: 2.7083 - val_accuracy: 0.0994 - val_loss: 1.0886 - learning_rate: 5.0000e-04
Epoch 4/75
23/23 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.3873 - loss: 1.4398 - val_accuracy: 0.0994 - val_loss: 1.1115 - learning_rate: 5.0000e-04
Epoch 5/75
23/23 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - accuracy: 0.4315 - loss: 1.0239 - val_accuracy: 0.2155 - val_loss: 1.1394 - learning_rate: 5.0000e-04
Epoch 6/75
23/23 ━━━━━━━━━━━━━━━━━━━━ 28s 973ms/step - accuracy: 0.4094 - loss